# Examples and Evaluation Metrics

We gradually develop any system we work on. For the classification task example, we:

1. Build baseline classifier
2. Build baseline evaluation
3. Do the optimization

The scope of this notebook is to explain and to show how to build the baseline evaluation and use it for further optimization.

In [15]:
import os

import dspy

In [5]:
lm = dspy.LM(
    model="openrouter/google/gemma-4-31b-it:free",
    api_base="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
dspy.settings.configure(lm=lm)

To set up a good evaluation process, we need two things: a good set of test data and a meaningful metric to evaluate how good each LM response is.

DSPy provides a small set of tools to help you streamline the evaluation process.

In [6]:
from datasets import load_dataset

In [7]:
ds = load_dataset("tuetschek/atis")
ds.set_format(type='pandas')
df_test = ds['test'][:]

ATIS_INTENT_MAPPING = {
   'abbreviation':                "Abbreviation and Fare Code Meaning Inquiry",
   'aircraft':                    "Aircraft Type Inquiry",
  #'aircraft+flight+flight_no':   "",
   'airfare':                     "Airfare and Fees Questions",
  #'airfare+flight_time':         "",
   'airline':                     "Airline Information Request",
  #'airline+flight_no':           "",
   'airport':                     "Airport Information and Queries",
   'capacity':                    "Aircraft Seating Capacity Inquiry",
   'Cheapest':                    "Cheapest Fare Inquiry",
   'city':                        "Airport Location Inquiry",
   'distance':                    "Airport Distance Inquiry",
   'flight':                      "Flight Booking Request",
  #'flight+airfare':              "",
   'flight_no':                   "Flight Number Inquiry",
   'flight_time':                 "Time Inquiry",
   'ground_fare':                 "Ground Transportation Cost Inquiry",
   'ground_service':              "Ground Transportation Inquiry",
  #'ground_service+ground_fare':  "Airport Ground Transportation and Cost Query",
   'meal':                        "Inquiry about In-flight Meals",
   'quantity':                    "Flight Quantity Inquiry",
   'restriction':                 "Flight Restriction Inquiry"
}
# Map the intents to their new names
df_test['intent'] = df_test['intent'].map(ATIS_INTENT_MAPPING)
df_test = df_test.dropna(subset=['intent'])
unique_intents = df_test['intent'].unique()

In [11]:
example_1 = dspy.Example(
   message = "I would like to find a flight from charlotte to las vegas that makes at "   
             "most one stop inbetween", 
   labels = unique_intents, 
   intent_label = "Flight Booking Request").with_inputs("message", "labels")

In [50]:
# Create for all examples we have in "test" dataset
def create_examples_from_set(set_name, n=-1):
   ds = load_dataset("tuetschek/atis")
   ds.set_format(type='pandas')
   df = ds[set_name][:]
   df['intent'] = df['intent'].map(ATIS_INTENT_MAPPING)
   df = df.dropna(subset='intent')
   unique_intents = df['intent'].unique()
   if n > 0:
      df = df.sample(n=n, random_state=42)
 
   examples = []
   for index in df.index:
       row = df.loc[index]
       examples.append(
           dspy.Example(message=row['text'],
                        labels=unique_intents.tolist(),
               intent_label=row['intent']).with_inputs('message', 'labels')
       )
   return examples
 
test_set = create_examples_from_set('test')

DSPy tends to assume train-validation-test splits.  
For LM-based applications, we might use splits like 15-70-15 or 20-30-50, which means only a small fraction of the data is used for training. With DSPy, it’s not always necessary to explicitly separate the training and validation data. Most optimizers accept just a training set and, if no validation set is provided, will automatically split it into training and validation sets. In general, DSPy recommends using about 30 to 300 examples for both training and validation.

In [51]:
import numpy as np

In [52]:
# We split "train" into train and validation sets
train_val_set = create_examples_from_set('train', n=200)

np.random.shuffle(train_val_set)
train_set = train_val_set[:20]
val_set = train_val_set[20:]

In [53]:
# We split "test" into dev and test sets
dev_test_set = create_examples_from_set('test', n=100)

np.random.shuffle(dev_test_set )
dev_set = dev_test_set [:50]
test_set = dev_test_set[50:]
# Yeah, it is confusing.

DSPy specifies a simple format to define your metric function. Metric functions must take three parameters: example, LM response, trace.

In [29]:
def validate_answer(example: dspy.Example, prediction: dspy.Prediction, trace=None):
   return example.intent_label == prediction.intent_label

In [31]:
# Since "validate_answer" is regular Python function, we can always test in unit tests or directly in the code
booking_flight_example = dspy.Example(
    message="I want to book a flight", 
    labels=unique_intents, 
    intent_label='Flight Booking Request').with_inputs("message", "labels")
 
booking_flight_prediction = dspy.Prediction(intent_label='Flight Booking Request')
 
print(validate_answer(booking_flight_example, booking_flight_prediction))

True


In [32]:
# We should also test with at least one negative example.
booking_flight_example = dspy.Example(
    message="I want to book a flight",
    labels=unique_intents,
    intent_label='Flight Booking Request').with_inputs("message", "labels")
 
booking_flight_prediction = dspy.Prediction(intent_label='Airport Distance Inquiry')
 
print(validate_answer(booking_flight_example, booking_flight_prediction))

False


# Evaluating the Baseline Model

DSPy program is `LM + Signature + Module + Optimization`  
However, the baseline program is only `LM + Signature + Module`

In [39]:
import time

In [56]:
len(dev_set)

50

In [57]:
class IntentSignature(dspy.Signature):
   """
   Classify the message into one or more of the possible intent labels.
   """
   message: str      = dspy.InputField()
   labels: list[str] = dspy.InputField()
   intent_label: str = dspy.OutputField(desc="Return the closest matches if any are "
      "reasonably close. Otherwise return None")

In [62]:
def evaluate_baseline(model="openrouter/google/gemma-4-31b-it:free", num_threads=10):
    lm = dspy.LM(
        model="openrouter/google/gemma-4-31b-it",
        api_base="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        cache=False
    )
    dspy.settings.configure(lm=lm)
    
    classifier = dspy.Predict(IntentSignature)
    evaluator = dspy.Evaluate(devset=dev_set,
                             num_threads=num_threads, 
                             display_progress=True, 
                             display_table=5, 
                             provide_traceback=True,
                             max_errors=5)
    
    start_time = time.time()
    results = evaluator(classifier, metric=validate_answer)
    end_time = time.time()
    
    print(f"total time: {end_time - start_time}")
    return results

In [63]:
eval_results = evaluate_baseline(num_threads=10)

Average Metric: 43.00 / 50 (86.0%): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:21<00:00,  2.29it/s]

2026/06/15 22:17:02 INFO dspy.evaluate.evaluate: Average Metric: 43 / 50 (86.0%)


,message,labels,example_intent_label,pred_intent_label,validate_answer
0,get fares from washington to boston,"['Flight Booking Request', 'Airfare and Fees Questions', 'Ground T...",Airfare and Fees Questions,Airfare and Fees Questions,✔️ [True]
1,list flights from las vegas to phoenix,"['Flight Booking Request', 'Airfare and Fees Questions', 'Ground T...",Flight Booking Request,Flight Booking Request,✔️ [True]
2,newark to tampa on friday,"['Flight Booking Request', 'Airfare and Fees Questions', 'Ground T...",Flight Booking Request,Flight Booking Request,✔️ [True]
3,i need flight numbers for those flights departing on thursday befo...,"['Flight Booking Request', 'Airfare and Fees Questions', 'Ground T...",Flight Number Inquiry,Flight Number Inquiry,✔️ [True]
4,new york city to las vegas and memphis to las vegas on sunday,"['Flight Booking Request', 'Airfare and Fees Questions', 'Ground T...",Flight Booking Request,Flight Booking Request,✔️ [True]


total time: 21.859060049057007


In [65]:
eval_results.score

86.0

In [68]:
eval_results.results[:2]

[(Example({'message': 'get fares from washington to boston', 'labels': ['Flight Booking Request', 'Airfare and Fees Questions', 'Ground Transportation Inquiry', 'Inquiry about In-flight Meals', 'Airport Information and Queries', 'Airline Information Request', 'Time Inquiry', 'Airport Location Inquiry', 'Ground Transportation Cost Inquiry', 'Flight Quantity Inquiry', 'Abbreviation and Fare Code Meaning Inquiry', 'Airport Distance Inquiry', 'Aircraft Type Inquiry', 'Aircraft Seating Capacity Inquiry', 'Flight Number Inquiry'], 'intent_label': 'Airfare and Fees Questions'}) (input_keys={'message', 'labels'}),
  Prediction(
      intent_label='Airfare and Fees Questions'
  ),
  True),
 (Example({'message': 'list flights from las vegas to phoenix', 'labels': ['Flight Booking Request', 'Airfare and Fees Questions', 'Ground Transportation Inquiry', 'Inquiry about In-flight Meals', 'Airport Information and Queries', 'Airline Information Request', 'Time Inquiry', 'Airport Location Inquiry', 'Gr

Overall score of the model is a good start. But with classification problems, it’s also useful to look at the model’s accuracy on each class.  
We often want to check for consistency. But it usually requires to go even beyond and re-run the same dataset multiple time and confirm that the scores do not deviate too much.

In [70]:
from sklearn.metrics import classification_report

In [71]:
predictions = []
groundtruths = []
for triplet in eval_results.results:
    example, prediction, score = triplet
    groundtruths.append(example.intent_label)
    predictions.append(prediction.intent_label.strip("'"))

In [75]:
print(classification_report(groundtruths, predictions))

                                                           precision    recall  f1-score   support

                                 "Flight Booking Request"       0.00      0.00      0.00         0
"Flight Booking Request", "Inquiry about In-flight Meals"       0.00      0.00      0.00         0
               Abbreviation and Fare Code Meaning Inquiry       1.00      1.00      1.00         3
                        Aircraft Seating Capacity Inquiry       1.00      1.00      1.00         1
                                    Aircraft Type Inquiry       0.50      1.00      0.67         1
                               Airfare and Fees Questions       1.00      0.50      0.67         4
                                 Airport Location Inquiry       0.00      0.00      0.00         1
                                   Flight Booking Request       0.94      0.89      0.92        37
                                    Flight Number Inquiry       1.00      1.00      1.00         1
         

/Users/dima/Projects/ai/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/dima/Projects/ai/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/dima/Projects/ai/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", resul

In [76]:
from sklearn.metrics import f1_score

In [77]:
f1_score(groundtruths, predictions, average='macro')

0.6041666666666666

Defining a metric function is typically the most difficult part of working with DSPy.